# Student Math Performance Predictor (G3 Final Grade)

## Mission and Problem
Empower underprivileged children through technology and digital education so they can thrive in a technology-driven world. Predict student final math outcomes (G3) early using learning and socio-demographic signals. Enable schools and community programs to identify at-risk learners before final exams. Support targeted, data-driven interventions that improve learning outcomes and long-term opportunity.

---

**Dataset:** [UCI Student Performance – Kaggle](https://www.kaggle.com/datasets/whenamancodes/student-performance) **File:** `student-mat.csv` **Target:** `G3` — final math grade (0–20) **Records:** 395 students × 33 features 

---
## 1. Imports & Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries loaded ')

---
## 2. Load Dataset

> **Download:** https://www.kaggle.com/datasets/whenamancodes/student-performance 

In [ ]:
csv_url = 'https://raw.githubusercontent.com/bianca255/linear_regression/main/linear_regression_model/summative/linear_regression/student-mat.csv'
df = pd.read_csv(csv_url, sep=';')

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values per Column:')
print(df.isnull().sum())
print('\nBasic Statistics (numerical):')
df.describe()

---
## 3. Exploratory Data Analysis (EDA)


In [ ]:
# ── Visualization 1: Target Variable Distribution ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['G3'], bins=21, range=(0, 20), color='steelblue',
 edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Final Math Grade (G3)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('G3 — Final Grade (0–20)')
axes[0].set_ylabel('Number of Students')
axes[0].axvline(df['G3'].mean(), color='crimson', linestyle='--',
 linewidth=1.8, label=f'Mean = {df["G3"].mean():.1f}')
axes[0].legend()

sns.boxplot(y=df['G3'], ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot of G3 — Final Grade', fontsize=14, fontweight='bold')
axes[1].set_ylabel('G3')

plt.tight_layout()
plt.savefig('plot_G3_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(f' Mean G3 = {df["G3"].mean():.2f}, Std = {df["G3"].std():.2f}')
print(' The distribution has a notable spike at 0, representing students who did not sit the final exam.')
print(' The bulk of students score between 8 and 16, with a slight left skew.')
print(' Students with G3 = 0 will be examined — these are strong at-risk signals.')

In [ ]:
# ── Key numerical feature distributions ────────────────────────────────────
num_cols = ['age', 'absences', 'studytime', 'failures', 'G1', 'G2',
 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'famrel']

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
 axes[i].hist(df[col], bins=15, color='teal', edgecolor='white', alpha=0.8)
 axes[i].set_title(col, fontsize=11, fontweight='bold')
 axes[i].set_xlabel(col)
 axes[i].set_ylabel('Count')

plt.suptitle('Distributions of Key Numerical Features', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(' G1 and G2 (period grades) are the most normally distributed — strong predictors of G3.')
print(' failures and absences are right-skewed: most students have none, but outliers matter.')
print(' studytime is low for most students, suggesting limited private study time.')

In [ ]:
# ── Scatter: G1, G2, studytime, absences vs G3 ─────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

scatter_features = ['G1', 'G2', 'studytime', 'absences']
colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for i, (col, color) in enumerate(zip(scatter_features, colors)):
 axes[i].scatter(df[col], df['G3'], alpha=0.4, color=color, s=25, edgecolors='none')
 # Trend line
 m, b = np.polyfit(df[col], df['G3'], 1)
 x_line = np.linspace(df[col].min(), df[col].max(), 100)
 axes[i].plot(x_line, m * x_line + b, color='black', linewidth=1.8, linestyle='--', label='Trend')
 axes[i].set_xlabel(col, fontsize=11)
 axes[i].set_ylabel('G3 — Final Grade', fontsize=11)
 axes[i].set_title(f'{col} vs G3', fontsize=12, fontweight='bold')
 axes[i].legend(fontsize=9)

plt.suptitle('Key Features vs Final Math Grade (G3)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_scatter_vs_G3.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(' G1 and G2 show a strong positive linear relationship with G3 — prior grades are the best predictors.')
print(' studytime shows a mild positive trend: more study correlates with slightly higher G3.')
print(' absences shows a slight negative trend: more absences weakly predict lower G3.')

---
## 4. Feature Engineering & Preprocessing


In [ ]:
# ── Identify categorical columns ──────────────────────────────────────────
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

for col in cat_cols:
 print(f' {col}: {df[col].unique().tolist()}')

In [ ]:
# ── Encode categorical columns to numeric ──────────────────────────────────
df_encoded = df.copy()
le = LabelEncoder()

for col in cat_cols:
 df_encoded[col] = le.fit_transform(df_encoded[col])
 print(f' Encoded "{col}"')

print(f'\nAll {len(cat_cols)} categorical columns encoded to numeric ')
df_encoded.dtypes

In [ ]:
# ── Visualization 2: Correlation Heatmap ──────────────────────────────────
corr = df_encoded.corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
 corr, mask=mask, annot=True, fmt='.2f',
 cmap='coolwarm', center=0, linewidths=0.4,
 annot_kws={'size': 7}, cbar_kws={'shrink': 0.75}
)
plt.title('Correlation Heatmap — All Features vs G3 (Final Grade)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(' G1 and G2 have the highest correlation with G3 (≈ 0.80 and 0.90 respectively).')
print(' failures correlates negatively (−0.36): more failures → lower final grade.')
print(' studytime and Medu (mother education) show small but positive correlations.')
print(' absences, Dalc (daily alcohol), and goout show weak negative correlations.')

In [ ]:
# ── Feature importance ranking by correlation with G3 ──────────────────────
target_corr = corr['G3'].drop('G3').abs().sort_values(ascending=False)

print('Feature correlation with G3 (absolute, sorted):')
print(target_corr.to_string())

In [ ]:
# ── Visualization 3: Top Feature Importances Bar Chart ────────────────────
top_n = 15
top_features = target_corr.head(top_n)

colors = ['#e74c3c' if v >= 0.3 else '#3498db' if v >= 0.1 else '#95a5a6'
 for v in top_features.values]

plt.figure(figsize=(11, 7))
bars = plt.barh(top_features.index[::-1], top_features.values[::-1],
 color=colors[::-1], edgecolor='white', height=0.65)
plt.xlabel('|Correlation with G3|', fontsize=12)
plt.title(f'Top {top_n} Feature Correlations with Final Grade (G3)', fontsize=14, fontweight='bold')
plt.axvline(x=0.3, color='crimson', linestyle='--', linewidth=1.4, alpha=0.7, label='Strong (≥ 0.3)')
plt.axvline(x=0.1, color='steelblue', linestyle='--', linewidth=1.4, alpha=0.7, label='Moderate (≥ 0.1)')
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('plot_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(' Red bars: strongly correlated features (|r| ≥ 0.3) — G2, G1, failures.')
print(' Blue bars: moderately correlated (0.1 ≤ |r| < 0.3) — Medu, studytime, absences.')
print(' Grey bars: weak signal — these can be dropped or kept as auxiliary context.')

In [ ]:
# ── Drop near-zero correlation features (|r| < 0.05) ─────────────────────
drop_cols = target_corr[target_corr < 0.05].index.tolist()
print(f'Dropping {len(drop_cols)} low-signal features: {drop_cols}')

df_model = df_encoded.drop(columns=drop_cols) if drop_cols else df_encoded.copy()
print(f'Remaining columns: {df_model.shape[1]} (including target G3)')

---
## 5. Train / Test Split & Standardization


In [ ]:
X = df_model.drop(columns=['G3'])
y = df_model['G3']

print(f'Features: {X.shape[1]} columns')
print(f'Feature names: {X.columns.tolist()}')
print(f'Target: G3 — {y.shape[0]} samples, range [{y.min()}, {y.max()}]')

# 80 / 20 split
X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42
)
print(f'\nTrain: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Persist scaler
joblib.dump(scaler, 'scaler.pkl')
print('\nFeatures standardized | scaler.pkl saved ')

---
## 6. Gradient Descent — Loss Curve (SGDRegressor)

> `SGDRegressor` is run epoch-by-epoch so we can record and plot the train and test MSE loss curve.


In [ ]:
train_losses = []
test_losses = []
n_epochs = 150

sgd_model = SGDRegressor(
 max_iter=1, tol=None, warm_start=True,
 learning_rate='invscaling', eta0=0.05,
 random_state=42
)

for epoch in range(n_epochs):
 sgd_model.fit(X_train_scaled, y_train)
 train_losses.append(mean_squared_error(y_train, sgd_model.predict(X_train_scaled)))
 test_losses.append( mean_squared_error(y_test, sgd_model.predict(X_test_scaled)))

print(f'Gradient Descent complete over {n_epochs} epochs')
print(f' Final Train MSE : {train_losses[-1]:.4f}')
print(f' Final Test MSE : {test_losses[-1]:.4f}')

In [ ]:
# ── Visualization 4: Loss Curve ───────────────────────────────────────────
plt.figure(figsize=(11, 5))
plt.plot(range(1, n_epochs + 1), train_losses,
 label='Train Loss (MSE)', color='steelblue', linewidth=2)
plt.plot(range(1, n_epochs + 1), test_losses,
 label='Test Loss (MSE)', color='coral', linewidth=2, linestyle='--')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error (MSE)', fontsize=12)
plt.title('Gradient Descent — Train vs Test Loss Curve', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('plot_loss_curve.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(' Both train and test losses drop sharply in the first ~20 epochs then plateau.')
print(' The train and test curves converge closely, indicating no significant overfitting.')
print(' The model generalizes well to unseen student data.')

---
## 7. Model 1 — Linear Regression (sklearn OLS)


In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

lr_train_mse = mean_squared_error(y_train, y_pred_lr_train)
lr_test_mse = mean_squared_error(y_test, y_pred_lr_test)
lr_test_r2 = r2_score(y_test, y_pred_lr_test)
lr_test_mae = mean_absolute_error(y_test, y_pred_lr_test)

print('=== Linear Regression ===')
print(f' Train MSE : {lr_train_mse:.4f}')
print(f' Test MSE : {lr_test_mse:.4f}')
print(f' Test MAE : {lr_test_mae:.4f}')
print(f' Test R² : {lr_test_r2:.4f}')

In [ ]:
# ── Visualization 5: Before / After Scatter Plot with Regression Line ──────
# Use G2 — the single strongest numerical predictor of G3
best_feature = 'G2'

X_single = df_encoded[[best_feature]].values
y_vals = df_encoded['G3'].values

X_single_sc = StandardScaler().fit_transform(X_single)
lr_single = LinearRegression().fit(X_single_sc, y_vals)
y_line = lr_single.predict(X_single_sc)

sort_idx = X_single_sc[:, 0].argsort()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# BEFORE — raw scatter, no line
axes[0].scatter(X_single, y_vals, alpha=0.4, color='steelblue', s=30,
 edgecolors='none', label='Students')
axes[0].set_xlabel(f'{best_feature} (Period 2 Grade)', fontsize=12)
axes[0].set_ylabel('G3 — Final Grade', fontsize=12)
axes[0].set_title(f'BEFORE — Raw Data: {best_feature} vs G3', fontsize=13, fontweight='bold')
axes[0].legend()

# AFTER — scatter + fitted regression line
axes[1].scatter(X_single, y_vals, alpha=0.4, color='steelblue', s=30,
 edgecolors='none', label='Students')
axes[1].plot(X_single[sort_idx], y_line[sort_idx],
 color='crimson', linewidth=2.5, label='Regression Line')
axes[1].set_xlabel(f'{best_feature} (Period 2 Grade)', fontsize=12)
axes[1].set_ylabel('G3 — Final Grade', fontsize=12)
axes[1].set_title(f'AFTER — Fitted Line: {best_feature} vs G3', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('plot_scatter_before_after.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n Interpretation:')
print(' The BEFORE plot shows the natural spread of G2 vs G3 — already visually linear.')
print(' The AFTER plot confirms the regression line captures the central trend well.')
print(' Students near 0 in G2 tend to also score near 0 in G3 — an early at-risk signal.')

---
## 8. Model 2 — Decision Tree Regressor


In [ ]:
dt_model = DecisionTreeRegressor(max_depth=6, min_samples_split=15,
 min_samples_leaf=5, random_state=42)
dt_model.fit(X_train_scaled, y_train)

y_pred_dt_train = dt_model.predict(X_train_scaled)
y_pred_dt_test = dt_model.predict(X_test_scaled)

dt_train_mse = mean_squared_error(y_train, y_pred_dt_train)
dt_test_mse = mean_squared_error(y_test, y_pred_dt_test)
dt_test_r2 = r2_score(y_test, y_pred_dt_test)
dt_test_mae = mean_absolute_error(y_test, y_pred_dt_test)

print('=== Decision Tree Regressor ===')
print(f' max_depth=6, min_samples_split=15')
print(f' Train MSE : {dt_train_mse:.4f}')
print(f' Test MSE : {dt_test_mse:.4f}')
print(f' Test MAE : {dt_test_mae:.4f}')
print(f' Test R² : {dt_test_r2:.4f}')

---
## 9. Model 3 — Random Forest Regressor


In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=10,
 min_samples_split=10, random_state=42,
 n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf_train = rf_model.predict(X_train_scaled)
y_pred_rf_test = rf_model.predict(X_test_scaled)

rf_train_mse = mean_squared_error(y_train, y_pred_rf_train)
rf_test_mse = mean_squared_error(y_test, y_pred_rf_test)
rf_test_r2 = r2_score(y_test, y_pred_rf_test)
rf_test_mae = mean_absolute_error(y_test, y_pred_rf_test)

print('=== Random Forest Regressor ===')
print(f' n_estimators=200, max_depth=10')
print(f' Train MSE : {rf_train_mse:.4f}')
print(f' Test MSE : {rf_test_mse:.4f}')
print(f' Test MAE : {rf_test_mae:.4f}')
print(f' Test R² : {rf_test_r2:.4f}')

---
## 10. Model Comparison & Save Best Model


In [ ]:
results = pd.DataFrame({
 'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
 'Train MSE': [lr_train_mse, dt_train_mse, rf_train_mse],
 'Test MSE': [lr_test_mse, dt_test_mse, rf_test_mse],
 'Test MAE': [lr_test_mae, dt_test_mae, rf_test_mae],
 'Test R²': [lr_test_r2, dt_test_r2, rf_test_r2],
})
results = results.sort_values('Test MSE').reset_index(drop=True)

print('='*55)
print('MODEL COMPARISON — sorted by Test MSE (lower = better)')
print('='*55)
print(results.to_string(index=False))
print(f'\n🏆 Best Model: {results.iloc[0]["Model"]}')

In [ ]:
# -- Visualization 6: Model Comparison Charts --
palette = ['#e74c3c', '#3498db', '#2ecc71']
model_names = results['Model'].tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('Test MSE', 'Test MSE - lower is better', False),
    ('Test MAE', 'Test MAE - lower is better', False),
    ('Test R²', 'Test R² - higher is better', True),
]

for ax, (metric, title, higher_better) in zip(axes, metrics):
    bars = ax.bar(model_names, results[metric], color=palette, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, results[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01 * results[metric].max(),
            f'{val:.2f}',
            ha='center',
            va='bottom',
            fontsize=9
        )

plt.suptitle('Model Comparison - G3 Prediction Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Save best model ────────────────────────────────────────────────────────
model_map = {
 'Linear Regression': lr_model,
 'Decision Tree': dt_model,
 'Random Forest': rf_model,
}
best_name = results.iloc[0]['Model']
best_model = model_map[best_name]

joblib.dump(best_model, 'best_model.pkl')

print(f' Best Model : {best_name}')
print(f' Test MSE : {results.iloc[0]["Test MSE"]:.4f}')
print(f' Test R² : {results.iloc[0]["Test R²"]:.4f}')
print(' Saved to : best_model.pkl')
print(' Scaler saved : scaler.pkl')

---
## 11. Single Prediction Script (for Task 2 — API)

Loads `best_model.pkl` + `scaler.pkl` and predicts G3 for one student from the test set.


In [ ]:
# ── Load persisted model & scaler ─────────────────────────────────────────
loaded_model = joblib.load('best_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

# ── Pick one sample from the test set ─────────────────────────────────────
sample_idx = 0
sample = X_test.iloc[[sample_idx]] # keep as DataFrame
actual_grade = y_test.iloc[sample_idx]

# ── Scale and predict ─────────────────────────────────────────────────────
sample_scaled = loaded_scaler.transform(sample)
predicted = loaded_model.predict(sample_scaled)[0]

print('=== Single Student Prediction ===')
print(f'Input features:')
for col, val in sample.iloc[0].items():
 print(f' {col:20s}: {val}')
print(f'\nActual G3 (final grade) : {actual_grade:.1f}')
print(f'Predicted G3 : {predicted:.2f}')
print(f'Absolute Error : {abs(actual_grade - predicted):.2f}')

---
## 12. Save Feature Columns (for Task 2 API)


In [ ]:
feature_cols = X.columns.tolist()

with open('feature_columns.json', 'w') as f:
 json.dump(feature_cols, f, indent=2)

print('feature_columns.json saved ')
print(f'Features ({len(feature_cols)}): {feature_cols}')